# MirrorTopology Step 1 — Phase D4C-1：正式 global 較正の **family partial**（1 family／1 run）launcher（notebook v0.2）
rules §9.4 の封印較正を family ごとに分割して実行する（design v0.2 §F.2；監査 R-D4DESIGN-D）。1 run＝1 family（E1／E2／E7／E8）：受入れ済み D-2 run（この family の `out/d2`：第 1 波 3 size × 3 position（E1 は 1 position）× 2 system の `cfg<id>_b0/b1/fit` と family reference `ref_<F>_b0/b1/fit`；`cfg*_w2` は読まない）と，非 E1 family では受入れ済み D-3b partition（3 size の `out/d3b`：追加 9 position）を **formal intake**（bytes／配列 SHA の再検証・spec・root SHA・manifest SHA＝登録 ledger の unit）で読み，D-3c の登録 plan を登録 ordered UID から再構築（identity＝受入れ値）し，12 位置入力を受入れ済み D-3c fingerprint に束縛し（`verify_consumer_family_inputs`），live の `twelve_official_gate`（keyword `plan_identity`／`table`）を通した上で，`step1_engine.d4c1_partial.calibrate_family_partial(mode='official')` を実行する：登録 D-2W W₂ context（`load_registered_w2_context`；context SHA＝pins）と登録 pseudo 列（`load_registered_pseudo_columns`；n＝2000・paired SHA＝pins）を registered loader から取り，**target の commitment（64 hex）だけ**を受け取る（nonce は notebook／source／log に残さない）。各 pseudo について共通 threshold evaluator（parent required quantities → position branch（W₂＝登録 context・event-ratio）→ coordinator → 12 位置段階）で eligible truth（technical_fail＞True＞unknown＞False）を導き，parent／plan／position source／12 位置完全 Result を content-addressed archive に保存し，partial record（schema `family_partial_calibration_v1`）を publication する。**partial は c／u／n・Wilson・usable・label を発行しない**（combiner が 4 family を集約する：別 notebook）。入力は読取り専用。**High-RAM runtime**（非 E1：第 1 波 18 配置＋12 位置 72 配置（第 1 波 18 は共有）≈10 GB＋plan ≈3.4 GB＋作業領域；E1 は標準 runtime で可）。時間は **`PROBE_N`（例 3）で先に実測**してから正式実行する（probe は正式 record にならない：`probe=True`，別 file 名，`campaign.id` に `__PROBE`）。

**attempt cell の規約（D-3c notebook v0.3／D4C-0 v0.2 と同一；R-D3C1-A/launch-boundary・B/output-anchor；R-D4C0-A/B）**：attempt cell は最初に **lock の出力先（anchor）** を検証し（symlink／非 directory／入力・source・stage root との交差／launcher 所有外 file／lock file の不一致なら何も書かずに停止），record はすべて anchor にだけ書く。設定・lock file・live source（実 HEAD・clean tree・inventory／script／pins／D-2・D-3b・D-3c・D4C-0 ledger／receipt／twelve asset の file SHA・engine 版）の検査は 1 つの検査関数にまとめ，**staging の前**と**子 process 起動の直前**の 2 回呼ぶ。attempt ID を発行し，危険な処理の前に「未完了・非成功」の initial record を `d4c1_partial_final_record.json` に書く（前回の final record は退避し，今回の証拠として再利用しない）。`partial_pass` は rc 0 ∧ `D4C1_PARTIAL_PASS` ∧ record の shape 正常 ∧ **信頼 REQUIRED inventory（lock に束縛した script の `REQUIRED_PARTIAL` を AST で抽出し notebook 定数と一致）の全 gate `is True`** ∧ attempt／lock／source／family／commitment／campaign への束縛一致 ∧ published partial record の SHA／bytes 一致と内容束縛（**JSON object でなければ失敗**；schema・family・mode official・commitment・pseudo n 2000＋列 SHA＝登録定数・context SHA＝pins・fingerprint at_gate＝at_end・per-pseudo 件数・**plan object の同一性記録**）∧ probe でない のときだけ True。科学的結果（eligible truth の分布）は PASS の条件ではない。

**v0.2（監査 R-D4C1-B／C）**：published record が JSON object でない（null／list／string／number）場合は evidence 失敗（検査を飛ばして成功しない）；REQUIRED 24 名（`G_plan_objects_stable`：第 1 波 view・組立 parent・12 位置 view の plan object が gate 前と較正後に同一（`is`）で family 共有であること；engine と script の両方で検査）。


In [1]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '231d37b89271793c8cd1b93c3282d2b851f83d3e'
EXPECTED_INVENTORY_SHA256 = '01f70420b40c556f119ae2c3c126c4ed51808bc5271429093d6cb15cc690b413'
FAMILY = 'E2'
D2_RUN_ROOT = '/content/drive/MyDrive/MirrorTopology_D2/E2_20260923T092030Z/out/d2'
D3B_RUN_ROOTS = {'L1.00': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.00_20260930T010656Z/out/d3b',
                 'L1.20': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.20_20260930T054800Z/out/d3b',
                 'L1.50': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.50_20260930T080141Z/out/d3b'}
TARGET_COMMITMENT = '9efa1b8d82fec2e23ac685d24dbaece7932e29955c94fc5a2955ba793c70b326'
CAMPAIGN_ID = 'D4C2_9efa1b8d82fe'
PROBE_N = 3                        # ← probe。None（正式）には変えない（本 GO の範囲外）
STAGE_LOCAL = True
STAGE_ROOT = '/content/stage'

In [2]:
# --- 1. fresh scratch checkout; inventory + script + pins + registered ledgers / receipts / assets bound; inputs exist and match the ledgers; Drive mounted (read-only usage); the LOCK (published, hashed)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and re.fullmatch(r'[0-9a-f]{64}', TARGET_COMMITMENT) and isinstance(CAMPAIGN_ID, str) and len(CAMPAIGN_ID)>=8 and '<' not in CAMPAIGN_ID
assert FAMILY in ('E1','E2','E7','E8') and (PROBE_N is None or (isinstance(PROBE_N, int) and not isinstance(PROBE_N, bool) and 1<=PROBE_N<2000))
assert (set(D3B_RUN_ROOTS)=={'L1.00','L1.20','L1.50'}) if FAMILY!='E1' else (D3B_RUN_ROOTS=={}), 'E2/E7/E8 need the three D-3b partitions; E1 takes none'
from google.colab import drive; drive.mount('/content/drive')
assert os.path.isfile(f'{D2_RUN_ROOT}/d2_bank_registry.json'), D2_RUN_ROOT
for s,p in D3B_RUN_ROOTS.items(): assert os.path.isfile(f'{p}/d3_bank_registry.json'), p
SCRATCH='/content/d4c1_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d4c1_calibration.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d4c1_calibration.py']
RA={k: f'{PHASEB}/{k}' for k in ('registered_assets/d2/d2_generation_ledger.json','registered_assets/d3b/d3b_generation_ledger.json','registered_assets/d3b/d3b_outer_receipt.json','registered_assets/d3c/d3c_profile_ledger.json','registered_assets/d3c/d3c_outer_receipt.json','registered_assets/d4c0/d4c0_ledger.json','registered_assets/d4c0/d4c0_outer_receipt.json','registered_assets/b3_2_twelve_assets.json','registered_assets/b3_2_shared_null_asset.json')}
PINS=f'{PHASEB}/d/d3_pins.json'; assert sha(PINS)==inv['d_sha256']['d/d3_pins.json'] and all(sha(p)==inv['registered_assets_sha256'][k] for k,p in RA.items()); RA_SHA={k: sha(p) for k,p in RA.items()}
d2l=json.load(open(RA['registered_assets/d2/d2_generation_ledger.json'])); assert D2_RUN_ROOT.rstrip('/')==d2l['families'][FAMILY]['run_root'].rstrip('/')+'/out/d2', (D2_RUN_ROOT, d2l['families'][FAMILY]['run_root'])
d3bl=json.load(open(RA['registered_assets/d3b/d3b_generation_ledger.json']))
for s,p in D3B_RUN_ROOTS.items(): assert p.rstrip('/')==d3bl['partitions'][f'{FAMILY}_{s}']['run_root'].rstrip('/')+'/d3b', (s, p, d3bl['partitions'][f'{FAMILY}_{s}']['run_root'])
pins=json.load(open(PINS)); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); RAM_INFO=dict(total_bytes=vm.total, available_bytes=vm.available); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1)); assert vm.total>(40e9 if FAMILY!='E1' else 10e9), 'E2/E7/E8 require a High-RAM runtime (>40 GB); E1 a standard runtime'
OUT=f'/content/d4c1_partial_{FAMILY}'
# OUTPUT-ANCHOR (R-D4C0-B; the lines from here to the end of this cell are re-used verbatim by the contract test): the output directory is validated BEFORE the first mkdir / lock publication
_in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
OWNED=re.compile(r'(d4c1_partial_lock\.json|d4c1_partial_final_record\.json|superseded_final_record_before_[0-9TZ]+_[0-9a-f]{10}\.json|run_[0-9TZ]+_[0-9a-f]{10}|d4c1_partial_(stdout|stderr)_[0-9TZ]+_[0-9a-f]{10}\.txt)')
def _protected_roots(): return [MT, PHASEB, PHASEC, STAGE_ROOT, '/content/drive', D2_RUN_ROOT, *D3B_RUN_ROOTS.values()]
def _safe_output_anchor(path, protected):
    # the ONE output policy (preflight and attempt): absolute path whose final component is not a symlink; disjoint from every protected root (source, Phase C, stage, Drive, inputs);
    # non-existent, or an existing real directory holding ONLY files owned by this launcher (lock, attempt records, run_* directories, stdout/stderr) and no symlinks. Nothing is written here.
    ap=os.path.abspath(path)
    if os.path.islink(ap) or os.path.realpath(ap)!=os.path.join(os.path.realpath(os.path.dirname(ap)), os.path.basename(ap)): raise RuntimeError(f'unsafe output: {path!r} is a symbolic link; nothing written')
    clash=[q for q in protected if _in(ap,q) or _in(q,ap)]
    if clash: raise RuntimeError(f'unsafe output: {path!r} overlaps a protected root {clash}; nothing written')
    if os.path.lexists(ap):
        if not os.path.isdir(ap): raise RuntimeError(f'unsafe output: {path!r} exists and is not a directory; nothing written')
        foreign=[e for e in sorted(os.listdir(ap)) if not OWNED.fullmatch(e) or os.path.islink(os.path.join(ap,e))]
        if foreign: raise RuntimeError(f'unsafe output: {path!r} holds entries not owned by this launcher {foreign[:5]}; nothing written')
    return ap
OUT=_safe_output_anchor(OUT, _protected_roots()); os.makedirs(OUT, exist_ok=True)      # attempts are separated inside OUT by their attempt id (nothing is deleted here)
# LOCK-BUILD
def _publish(path, doc):
    data=json.dumps(doc, indent=1, ensure_ascii=False, default=str).encode('utf-8'); tmp=path+'.tmp'; open(tmp,'wb').write(data); os.replace(tmp, path); assert open(path,'rb').read()==data, path; return hashlib.sha256(data).hexdigest()
lock=dict(schema='d4c1_partial_launcher_lock_v1', commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv['d_sha256']['d/d4c1_calibration.py'], pins_sha256=sha(PINS), registered_assets_sha256=RA_SHA, engine=inv['engine_version'],
          family=FAMILY, d2_run_root=D2_RUN_ROOT, d3b_run_roots=dict(D3B_RUN_ROOTS), target_commitment=TARGET_COMMITMENT, campaign_id=CAMPAIGN_ID, probe_n=PROBE_N, stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT, locked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
LOCK_PATH=f'{OUT}/d4c1_partial_lock.json'; LOCK_SHA256=_publish(LOCK_PATH, lock); print('lock', LOCK_SHA256); print({k: v for k, v in lock.items() if k != 'registered_assets_sha256'})


Mounted at /content/drive
RAM total GB 54.8 available GB 52.6
lock 9632f840b15c32d0f7f2638e5655f3c5332153b2e1e4b4c999cb3240a94eaa49
{'schema': 'd4c1_partial_launcher_lock_v1', 'commit': '231d37b89271793c8cd1b93c3282d2b851f83d3e', 'inventory_sha256': '01f70420b40c556f119ae2c3c126c4ed51808bc5271429093d6cb15cc690b413', 'script_sha256': '834ab646ae4617e6a87bfcbea8eb112a818254f92786d7da65c2ba93fabef1fa', 'inventory_script_sha256': '834ab646ae4617e6a87bfcbea8eb112a818254f92786d7da65c2ba93fabef1fa', 'pins_sha256': '2cd9cf0105004b75e396807f5bd455e5cdebda64cc8bdd20c08d4e5062babed3', 'engine': '0.107.0', 'family': 'E2', 'd2_run_root': '/content/drive/MyDrive/MirrorTopology_D2/E2_20260923T092030Z/out/d2', 'd3b_run_roots': {'L1.00': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.00_20260930T010656Z/out/d3b', 'L1.20': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.20_20260930T054800Z/out/d3b', 'L1.50': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.50_20260930T080141Z/out/d3b'}, 'target_commi

In [3]:
# --- 2. ONE ATTEMPT (D-3c v0.3 pattern; R-D3C1-A/launch-boundary + R-D3C1-B/output-anchor; R-D4C0-A/B; R-D4DESIGN-G): (0) the SAFE anchored output is taken from the LOCK and validated BEFORE anything is written;
#        (1) attempt id + initial NON-success record at the anchor; (2) ONE verifier (configuration, lock file, LIVE source) runs BEFORE staging; (3) staging (disk check first; first-wave D-2 units + D-3b partitions);
#        (4) the SAME verifier runs again IMMEDIATELY before subprocess.run (no child process on any mismatch); (5) record validation (trusted REQUIRED inventory) + binding + published partial record content AND semantic check; final record at the anchor only
t0=time.time(); _in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
ANCHOR=lock['out']
try:
    if '_safe_output_anchor' not in globals(): raise RuntimeError('preflight cell not run')
    _safe_output_anchor(ANCHOR, [lock['mt'], lock['phaseb'], lock['phasec'], lock['stage_root'], '/content/drive', lock['d2_run_root'], *lock['d3b_run_roots'].values(), *_protected_roots()])
    if not os.path.isdir(ANCHOR) or LOCK_PATH!=f'{ANCHOR}/d4c1_partial_lock.json' or not os.path.isfile(LOCK_PATH): raise RuntimeError('lock path not inside the anchor')      # the lock file's CONTENT is verified by the verifier below (a FAIL record at this anchor)
except Exception as _anchor_error: print(f'ATTEMPT NOT STARTED: the locked output {ANCHOR!r} is not a safe anchored directory ({_anchor_error}); nothing was written', file=sys.stderr); raise RuntimeError('unsafe output anchor; nothing written')
ATTEMPT=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())+'_'+hashlib.sha256(f'{time.time_ns()}|D4C1P|{REPO_COMMIT}|{LOCK_SHA256}'.encode()).hexdigest()[:10]
FIN=f'{ANCHOR}/d4c1_partial_final_record.json'; RUN=f'{ANCHOR}/run_{ATTEMPT}'; partial_pass=False; rc=None; v=None; PROBE=lock['probe_n'] is not None; TAG=('probe_' if PROBE else 'partial_')+lock['family']
REC=dict(schema='d4c1_partial_launcher_final_record_v1', attempt_id=ATTEMPT, started_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), lock_sha256=LOCK_SHA256, lock=lock, output_anchor=ANCHOR, ram=globals().get('RAM_INFO'), stage='started', partial_pass=False, probe=PROBE, exit_code=None, D4C1_PARTIAL_PASS=None, launcher_fallback=False, gates_ok=None, bindings_ok=None, evidence_ok=None, failures=[], exception=None, run_dir=RUN, staged_inputs=None, bindings={}, seconds_total=None, finished_utc=None)
if os.path.exists(FIN): prev=f'{ANCHOR}/superseded_final_record_before_{ATTEMPT}.json'; os.replace(FIN, prev); REC['superseded_previous_record']=prev      # a previous attempt's record is NEVER this attempt's evidence
_publish(FIN, REC)
def _finish(stage, failures=(), exc=None):
    REC.update(stage=stage, exception=(repr(exc) if exc is not None else None), seconds_total=round(time.time()-t0, 3), finished_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())); REC['failures']+=list(failures); REC['partial_pass']=bool(partial_pass); _publish(FIN, REC); print('attempt', ATTEMPT, '| stage', stage, '| partial_pass', REC['partial_pass'], '| probe', PROBE, '| failures', REC['failures'])
def _verify_locked_state(prefix):
    # the ONE verifier: current configuration == lock; lock file on disk == lock; LIVE source (actual HEAD, clean tree, file SHAs of inventory / script / pins / registered ledgers, receipts and assets,
    # the inventory's script SHA, engine version, tree layout) == lock; input roots present. Called before staging and again immediately before the child launch.
    REC['stage']=f'{prefix}_config'; cfg=dict(family=FAMILY, d2_run_root=D2_RUN_ROOT, d3b_run_roots=dict(D3B_RUN_ROOTS), target_commitment=TARGET_COMMITMENT, campaign_id=CAMPAIGN_ID, probe_n=PROBE_N, stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, commit=REPO_COMMIT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT)
    mism={k: (cfg[k], lock.get(k)) for k in cfg if cfg[k]!=lock.get(k)}
    if mism: raise RuntimeError(f'configuration differs from the lock (re-run the preflight cell): {mism}')
    if LOCK_PATH!=f'{ANCHOR}/d4c1_partial_lock.json' or sha(LOCK_PATH)!=LOCK_SHA256 or json.load(open(LOCK_PATH))!=lock: raise RuntimeError('lock file on disk differs from the lock in memory')
    REC['stage']=f'{prefix}_source'; inv_now=json.load(open(INV))
    src=dict(head=subprocess.check_output(['git','-C',MT,'rev-parse','HEAD'],text=True).strip(), clean=(subprocess.check_output(['git','-C',MT,'status','--porcelain'],text=True).strip()==''), inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv_now['d_sha256']['d/d4c1_calibration.py'], pins_sha256=sha(PINS), registered_assets_sha256={k: sha(p) for k,p in RA.items()}, engine=inv_now['engine_version'], checked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    want=dict(head=lock['commit'], clean=True, inventory_sha256=lock['inventory_sha256'], script_sha256=lock['script_sha256'], inventory_script_sha256=lock['script_sha256'], pins_sha256=lock['pins_sha256'], registered_assets_sha256=lock['registered_assets_sha256'], engine=lock['engine'])
    bad=[k for k in want if src[k]!=want[k]]; REC['bindings'][f'live_source_{prefix}']=src
    if bad or PHASEB!=f'{MT}/engine/phaseB' or PHASEC!=f'{MT}/phaseC' or SCRIPT!=f'{PHASEB}/d/d4c1_calibration.py': raise RuntimeError(f'live source differs from the lock ({prefix}): {bad}')
    missing=[p for p in [D2_RUN_ROOT, *D3B_RUN_ROOTS.values()] if not os.path.isdir(p)]
    if missing: raise RuntimeError(f'input roots missing: {missing}')
try:
    # 2. configuration / lock file / live source BEFORE staging (editing cell 0 after the preflight, or skipping the preflight, is refused here)
    _verify_locked_state('precheck'); _publish(FIN, REC)
    # 3. optional local staging: the D-2 registry + first-wave units (cfg<id>_b0/b1/fit of this family's first-wave configurations and ref_<F>_b0/b1/fit; cfg*_w2 excluded) and the whole D-3b partitions;
    #    disk capacity is checked BEFORE the copy; the script re-verifies every byte / array SHA of the staged copy
    REC['stage']='staging'; IN_D2=D2_RUN_ROOT; IN_D3B=dict(D3B_RUN_ROOTS)
    if STAGE_LOCAL:
        ST=STAGE_ROOT; guard=[q for q in [D2_RUN_ROOT, *D3B_RUN_ROOTS.values(), ANCHOR, MT] if _in(ST,q) or _in(q,ST)]
        if guard: raise RuntimeError(f'STAGE_ROOT must be disjoint from the inputs, the output and the source: {guard}')
        units=sorted(u for u in os.listdir(D2_RUN_ROOT) if (re.fullmatch(r'cfg\d+_(b0|b1|fit)', u) or re.fullmatch(r'ref_'+FAMILY+r'_(b0|b1|fit)', u)) and os.path.isdir(os.path.join(D2_RUN_ROOT,u)))
        dsize=lambda root: sum(os.path.getsize(os.path.join(d,x)) for d,_,fs in os.walk(root) for x in fs)
        need=sum(dsize(os.path.join(D2_RUN_ROOT,u)) for u in units)+os.path.getsize(os.path.join(D2_RUN_ROOT,'d2_bank_registry.json'))+sum(dsize(p) for p in D3B_RUN_ROOTS.values()); ts=time.time()
        shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST); free=shutil.disk_usage(ST).free; REC['staging']=dict(root=ST, d2_units=len(units), d3b_partitions=sorted(D3B_RUN_ROOTS), input_bytes=need, disk_free_bytes_before=free)
        if free < need*1.05+2e9: raise RuntimeError(f'insufficient local disk for staging: need {need/1e9:.1f} GB (+5% +2 GB margin), free {free/1e9:.1f} GB')
        IN_D2=f'{ST}/d2'; os.makedirs(IN_D2); shutil.copy2(os.path.join(D2_RUN_ROOT,'d2_bank_registry.json'), os.path.join(IN_D2,'d2_bank_registry.json'))
        for u in units: shutil.copytree(os.path.join(D2_RUN_ROOT,u), os.path.join(IN_D2,u))
        IN_D3B={s: f'{ST}/d3b_{s}' for s in D3B_RUN_ROOTS}
        for s,p in D3B_RUN_ROOTS.items(): shutil.copytree(p, IN_D3B[s])
        REC['staging'].update(seconds=round(time.time()-ts,1), disk_free_bytes_after=shutil.disk_usage(ST).free); print('staged in', round(time.time()-ts), 's; disk free GB', round(REC['staging']['disk_free_bytes_after']/1e9,1))
    REC['staged_inputs']=dict(d2=IN_D2, d3b=IN_D3B); _publish(FIN, REC)
    # 4. the SAME verifier IMMEDIATELY before the launch (state may have changed during the copy): any mismatch -> FAIL record, NO child process
    _verify_locked_state('prelaunch')
    REC['stage']='launch'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',RUN,'--mode','partial','--family',FAMILY,'--d2-root',IN_D2,'--target-commitment',TARGET_COMMITMENT,'--campaign-id',CAMPAIGN_ID,'--profile','production_official','--attempt-id',ATTEMPT,'--launcher-lock-sha256',LOCK_SHA256]+[x for s in sorted(IN_D3B) for x in ('--d3b-root', f'{s}={IN_D3B[s]}')]+([] if not PROBE else ['--probe-n', str(PROBE_N)])
    REC['args']=args; _publish(FIN, REC); rc=subprocess.run(args,capture_output=True,text=True); REC['exit_code']=rc.returncode
    open(f'{ANCHOR}/d4c1_partial_stdout_{ATTEMPT}.txt','w').write(rc.stdout); open(f'{ANCHOR}/d4c1_partial_stderr_{ATTEMPT}.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
    # 5. semantic validation of the record (R-D4C0-A): exact schema / complete stage / empty failures / formal profile / no self-test (shape -> launcher_fallback); the TRUSTED REQUIRED inventory
    #    (notebook constant == REQUIRED_PARTIAL extracted by AST from the lock-bound script) with every gate `is True` (-> gates_ok); binding to THIS attempt / lock / source / family / commitment / campaign
    #    (-> bindings_ok); the published partial record content-verified AND semantically bound (schema, family, mode, commitment, registered pseudo identity, context SHA = pins, fingerprints, inventory) (-> evidence_ok)
    REC['stage']='record'; vp=f'{RUN}/d4c1_{TAG}_run.json'; shape=[]; gates_bad=[]; binding=[]; evidence=[]
    REQUIRED_EXPECTED=['G_pins_loaded','G_engine_inventory','G_script_sha','G_phaseC_members','G_env_lock','G_external_loader_sha','G_twelve_context','G_w2_context_registered','G_pseudo_registered','G_twelve_asset_registered','G_commitment_form','G_d3b_units_accepted','G_inputs_resolved','G_first_wave_supplies','G_plans_fixed','G_first_wave_views','G_twelve_inputs','G_twelve_gate','G_partial_computed','G_plan_objects_stable','G_partial_published','G_partial_verified','G_pseudo_identity_bound','G_record_saved']
    import ast as _ast
    _req_src=[list(_ast.literal_eval(st.value)) for st in _ast.parse(open(SCRIPT).read()).body if isinstance(st,_ast.Assign) and any(isinstance(t,_ast.Name) and t.id=='REQUIRED_PARTIAL' for t in st.targets)]
    if _req_src!=[REQUIRED_EXPECTED]: gates_bad.append(f'REQUIRED_PARTIAL inventory of the lock-bound script differs from the trusted constant: {_req_src}')
    try: v=json.load(open(vp))
    except Exception as read_error: v=None; shape.append(f'run record unreadable: {read_error!r}')
    if v is not None and not isinstance(v, dict): shape.append('run record is not a dictionary'); v=None
    if v is not None:
        num=lambda x: isinstance(x,(int,float)) and not isinstance(x,bool)
        for k, ok in [('schema', v.get('schema')=='d4c1_run_record_v1'), ('mode', v.get('mode')=='partial'), ('family', v.get('family')==FAMILY), ('D4C1_PARTIAL_PASS', isinstance(v.get('D4C1_PARTIAL_PASS'), bool)), ('stage', v.get('stage')=='complete'), ('gates', isinstance(v.get('gates'), dict)), ('required_inventory', isinstance(v.get('required_inventory'), list)), ('required_all_true', isinstance(v.get('required_all_true'), bool)), ('failures', v.get('failures')==[]), ('seconds', num(v.get('seconds'))), ('stages_rss_mb', isinstance(v.get('stages_rss_mb'), dict)),
                      ('attempt', isinstance(v.get('attempt'), dict)), ('source', isinstance(v.get('source'), dict)), ('selftest', v.get('selftest') is False), ('formal', v.get('formal') is True), ('probe', v.get('probe') is PROBE), ('profile', v.get('profile')=='production_official'), ('partial', isinstance(v.get('partial'), dict)), ('pseudo', isinstance(v.get('pseudo'), dict)), ('w2_context', isinstance(v.get('w2_context'), dict)), ('published_evidence', isinstance(v.get('published_evidence'), dict))]:
            if not ok: shape.append(f'run record field invalid: {k}')
        g=v.get('gates') if isinstance(v.get('gates'), dict) else {}
        if sorted(g)!=sorted(REQUIRED_EXPECTED): gates_bad.append(f'gate set differs from the trusted REQUIRED inventory: {sorted(g)}')
        if v.get('required_inventory')!=REQUIRED_EXPECTED: gates_bad.append('required_inventory differs from the trusted REQUIRED inventory')
        if not all(g.get(k) is True for k in REQUIRED_EXPECTED): gates_bad.append(f'required gates not all True: {[k for k in REQUIRED_EXPECTED if g.get(k) is not True]}')
        if v.get('required_all_true') is not True: gates_bad.append('required_all_true is not True')
        a_=v.get('attempt') if isinstance(v.get('attempt'), dict) else {}; s_=v.get('source') if isinstance(v.get('source'), dict) else {}
        for k, ok in [('attempt_id', a_.get('attempt_id')==ATTEMPT), ('launcher_lock_sha256', a_.get('launcher_lock_sha256')==LOCK_SHA256), ('source.script_sha256', s_.get('script_sha256')==lock['script_sha256']), ('source.inventory_sha256', s_.get('inventory_sha256')==lock['inventory_sha256']), ('source.pins_sha256', s_.get('pins_sha256')==lock['pins_sha256']), ('source.engine_version', s_.get('engine_version')==lock['engine']), ('target_commitment', v.get('target_commitment')==lock['target_commitment']), ('campaign_id', v.get('campaign_id')==lock['campaign_id']), ('probe_n', v.get('probe_n')==lock['probe_n'])]:
            if not ok: binding.append(f'run record not bound to this attempt / lock: {k}')
        pe=v.get('published_evidence') if isinstance(v.get('published_evidence'), dict) else {}; fn=f'd4c1_{TAG}_record.json'; fp=f'{RUN}/{fn}'; e=pe.get(fn); ps_=v.get('pseudo') if isinstance(v.get('pseudo'), dict) else {}; idn=ps_.get('identity') if isinstance(ps_.get('identity'), dict) else {}; w2_=v.get('w2_context') if isinstance(v.get('w2_context'), dict) else {}
        if set(pe)!={fn}: evidence.append(f'published evidence set {sorted(pe)} != {[fn]}')
        if not (isinstance(e, dict) and os.path.isfile(fp) and not os.path.islink(fp) and sha(fp)==e.get('sha256') and os.path.getsize(fp)==e.get('bytes')): evidence.append(f'published partial record not content-verified: {fn}')
        else:
            try: doc=json.load(open(fp)); doc_ok=True
            except Exception as doc_error: doc=None; doc_ok=False; evidence.append(f'published partial record unreadable: {doc_error!r}')
            if doc_ok and not isinstance(doc, dict): evidence.append(f'published partial record is not a JSON object ({type(doc).__name__}); its content cannot be bound')      # R-D4C1-B: null / list / string / number are failures, never skipped
            if doc_ok and isinstance(doc, dict):
                thr=doc.get('thresholds') if isinstance(doc.get('thresholds'), dict) else {}; pp=thr.get('pseudo') if isinstance(thr.get('pseudo'), dict) else {}; fps=doc.get('fingerprints') if isinstance(doc.get('fingerprints'), dict) else {}; part=v.get('partial') if isinstance(v.get('partial'), dict) else {}; n_rows=(lock['probe_n'] if PROBE else 2000)
                checks=[('schema', doc.get('schema')=='family_partial_calibration_v1' and doc.get('kind')=='family_partial_calibration'), ('family', doc.get('family')==FAMILY), ('mode', doc.get('mode')==('smoke' if PROBE else 'official')), ('engine', doc.get('engine_version')==lock['engine']), ('commitment', thr.get('target')is None and thr.get('target_commitment')==lock['target_commitment']),
                        ('pseudo_n', pp.get('n')==n_rows and len(doc.get('per_pseudo_status') or [])==n_rows), ('pseudo_identity', PROBE or (pp.get('sha256_T1')==idn.get('T1_sha256') and pp.get('sha256_T2')==idn.get('T2_sha256') and idn.get('paired_sha256')==pins['pseudo_paired_sha256'])), ('context_sha', doc.get('w2_context_sha256')==pins['d2w_context_sha256']==w2_.get('context_sha256')),
                        ('fingerprints', isinstance(fps.get('at_gate'), dict) and fps.get('at_gate')==fps.get('at_end') and FAMILY in fps['at_gate'] and all(f'{FAMILY}/{sz}' in fps['at_gate'] for sz in ('L1.00','L1.20','L1.50')) and (FAMILY=='E1' or all(f'twelve:{FAMILY}/{sz}' in fps['at_gate'] for sz in ('L1.00','L1.20','L1.50')))), ('gate', isinstance(doc.get('gate'), dict) and doc['gate'].get('passed') is True and doc['gate'].get('mode')==('smoke' if PROBE else 'official')),
                        ('no_aggregation', 'calibration' not in doc and 'families' not in doc and 'cases' not in doc), ('plan_objects', isinstance(doc.get('plan_objects'), dict) and doc['plan_objects'].get('stable_after') is True and doc['plan_objects'].get('first_wave_shared') is True and (FAMILY=='E1' or doc['plan_objects'].get('twelve_shares_first_wave') is True)), ('partial_sha', isinstance(doc.get('binding'), dict) and doc['binding'].get('partial_sha256')==part.get('partial_sha256') and isinstance(part.get('verification'), dict) and part['verification'].get('ok') is True and part['verification'].get('registered_context_ok') is True), ('campaign', isinstance(doc.get('campaign'), dict) and doc['campaign'].get('id')==(lock['campaign_id']+('__PROBE' if PROBE else ''))), ('archive_index', os.path.isfile(f'{RUN}/archive/index.json'))]
                for k, ok in checks:
                    if not ok: evidence.append(f'published partial record not bound: {k}')
        REC.update(D4C1_PARTIAL_PASS=v.get('D4C1_PARTIAL_PASS'), record_sha256=sha(vp), record_stage=v.get('stage'), record_failures=v.get('failures'), gates=v.get('gates'), stages_rss_mb=v.get('stages_rss_mb'), stages_peak_rss_mb=v.get('stages_peak_rss_mb'), seconds_script=v.get('seconds'), timings=v.get('timings'), partial_summary={k: (v.get('partial') or {}).get(k) for k in ('partial_sha256','n_pseudo','eligibility_counts','expanded','twelve_evaluated','archive_entries')})
        print('rc', rc.returncode, '| stage', v.get('stage'), '| D4C1_PARTIAL_PASS', v.get('D4C1_PARTIAL_PASS'), '| failures', v.get('failures'), '| peak rss MB', v.get('stages_peak_rss_mb'), '| timings', v.get('timings')); print('partial', REC['partial_summary'])
    REC.update(launcher_fallback=bool(shape), gates_ok=(not gates_bad) if v is not None else None, bindings_ok=(not binding) if v is not None else None, evidence_ok=(not evidence) if v is not None else None, stderr_tail=rc.stderr[-2000:])
    partial_pass=bool(rc.returncode==0 and v is not None and v.get('D4C1_PARTIAL_PASS') is True and not PROBE and not shape and not gates_bad and not binding and not evidence)      # the scientific outcome (eligible truth distribution) is NOT a condition; a probe is never a pass
    _finish('complete', shape+gates_bad+binding+evidence)
except Exception as attempt_error:
    partial_pass=False; _finish(REC.get('stage') or 'unknown', [f'exception at stage {REC.get("stage")}: {attempt_error!r}'], attempt_error)
print('partial_pass', partial_pass)


staged in 846 s; disk free GB 195.2
  first-wave intake 20101 (L1.00) 8s rss 1707 MB
  first-wave intake 20102 (L1.00) 6s rss 2143 MB
  first-wave intake 20103 (L1.00) 6s rss 2431 MB
  first-wave intake 20201 (L1.20) 6s rss 2715 MB
  first-wave intake 20202 (L1.20) 6s rss 3023 MB
  first-wave intake 20203 (L1.20) 6s rss 3306 MB
  first-wave intake 20301 (L1.50) 6s rss 3590 MB
  first-wave intake 20302 (L1.50) 6s rss 3873 MB
  first-wave intake 20303 (L1.50) 6s rss 4181 MB
  twelve intake 20104 (L1.00 position 3) 8s rss 7845 MB
  twelve intake 20105 (L1.00 position 4) 8s rss 8128 MB
  twelve intake 20106 (L1.00 position 5) 8s rss 8412 MB
  twelve intake 20107 (L1.00 position 6) 8s rss 8695 MB
  twelve intake 20108 (L1.00 position 7) 8s rss 8979 MB
  twelve intake 20109 (L1.00 position 8) 8s rss 9263 MB
  twelve intake 20110 (L1.00 position 9) 8s rss 9554 MB
  twelve intake 20111 (L1.00 position 10) 8s rss 9874 MB
  twelve intake 20112 (L1.00 position 11) 8s rss 10157 MB
  twelve intake 

In [4]:
# --- 3. zip the evidence (lock, attempt records (current + superseded), run record, published partial record, archive, stdout/stderr) for the audit, the combination and registration
zp=f'/content/d4c1_partial_{lock["family"]}_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=lock['out'])
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)


/content/d4c1_partial_E2_231d37b89271.zip 515915


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>